# Virtual Staining of Embryo Images
## BF → IF (DAPI + Phalloidin) Regression

**Approaches:**
1. TransUNet — CNN encoder + ViT + U-Net decoder with skip connections
2. SwinUNet — Window attention (better at local details)
3. Pix2PixHD — Conditional GAN with multi-scale discriminator
4. Conditional DDPM — Diffusion-based (highest fidelity, slowest)
5. AdaIN/EFDM — Style transfer (also for fixed→live BF domain adaptation)

**Runtime:** Use GPU (T4 minimum, A100/V100 recommended for diffusion)

---
## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repository
!git clone -b claude/embryo-virtual-staining-j4Kfg https://github.com/erkankalafat/virtual-strain-embryos.git
%cd virtual-strain-embryos

In [ ]:
# Install dependencies
!pip install -q tifffile einops ml-collections pyyaml scikit-image

In [ ]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

---
## 2. Configure Data Path

Upload your embryo folders to Google Drive in this structure:
```
My Drive/
  embryo_data/
    project_Embryo1_.../
      BF/
      DAPI/
      Phalloidin/
      DAPI+Phalloidin/
      Overlay/
    project_Embryo2_.../
      BF/
      ...
```

**Set the path below to match your Drive folder:**

In [ ]:
#@title Data Configuration { display-mode: "form" }

#@markdown **Path to your embryo data folder on Google Drive:**
DATA_ROOT = "/content/drive/MyDrive/embryo_data"  #@param {type:"string"}

#@markdown **Target channels to predict:**
TARGET_CHANNELS = "both"  #@param ["both", "dapi", "phalloidin", "combined", "overlay"]

#@markdown **Image size (512 for regression models, 256 for diffusion):**
IMG_SIZE = 512  #@param {type:"integer"}

#@markdown **Batch size (reduce if OOM):**
BATCH_SIZE = 4  #@param {type:"integer"}

import os
assert os.path.exists(DATA_ROOT), f"Data path not found: {DATA_ROOT}\nCheck your Google Drive path."

# List discovered project folders
projects = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)) and not d.startswith('.')]
print(f"Found {len(projects)} project folder(s):")
for p in sorted(projects):
    subfolders = [s for s in os.listdir(os.path.join(DATA_ROOT, p)) if os.path.isdir(os.path.join(DATA_ROOT, p, s))]
    print(f"  {p}/  →  {subfolders}")

In [ ]:
# Test the dataset loader
from src.data import EmbryoDataset

ds = EmbryoDataset(
    root_dir=DATA_ROOT,
    target_channels=TARGET_CHANNELS,
    img_size=IMG_SIZE,
    augment=False,
)
print(f"Total paired images found: {len(ds)}")

# Preview a sample
sample = ds[0]
print(f"BF shape: {sample['bf'].shape}  (should be [3, {IMG_SIZE}, {IMG_SIZE}])")
print(f"Target shape: {sample['target'].shape}")
print(f"BF range: [{sample['bf'].min():.3f}, {sample['bf'].max():.3f}]")
print(f"Target range: [{sample['target'].min():.3f}, {sample['target'].max():.3f}]")

In [ ]:
# Visualize a few BF-IF pairs
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(min(3, len(ds)), 4, figsize=(16, 4 * min(3, len(ds))))
if min(3, len(ds)) == 1:
    axes = axes[np.newaxis, :]

for i in range(min(3, len(ds))):
    sample = ds[i]
    bf = sample['bf'].permute(1, 2, 0).numpy()
    tgt = sample['target']

    axes[i, 0].imshow(bf[:, :, 0], cmap='gray')
    axes[i, 0].set_title(f'BF ({sample["name"]})')

    if tgt.shape[0] >= 1:
        axes[i, 1].imshow(tgt[0].numpy(), cmap='Blues')
        axes[i, 1].set_title('DAPI')

    if tgt.shape[0] >= 2:
        axes[i, 2].imshow(tgt[1].numpy(), cmap='Reds')
        axes[i, 2].set_title('Phalloidin')

    # Composite: DAPI=blue, Phalloidin=red
    if tgt.shape[0] >= 2:
        composite = np.zeros((*tgt.shape[1:], 3))
        composite[:, :, 2] = tgt[0].numpy()  # Blue = DAPI
        composite[:, :, 0] = tgt[1].numpy()  # Red = Phalloidin
        axes[i, 3].imshow(composite)
        axes[i, 3].set_title('DAPI+Phalloidin')
    else:
        axes[i, 2].axis('off')
        axes[i, 3].axis('off')

    for ax in axes[i]:
        ax.axis('off')

plt.tight_layout()
plt.show()

---
## 3. Helper: Build Config

Instead of editing YAML files, we build configs programmatically below.

In [ ]:
def make_config(model_name, **overrides):
    """Build a training config dict for a given model."""
    import yaml
    with open(f'configs/{model_name}.yaml', 'r') as f:
        config = yaml.safe_load(f)

    # Override data settings with Colab values
    config['data']['root_dir'] = DATA_ROOT
    config['data']['target_channels'] = TARGET_CHANNELS
    config['data']['img_size'] = IMG_SIZE
    config['data']['batch_size'] = BATCH_SIZE
    config['data']['num_workers'] = 2  # Colab has limited workers

    # Apply any extra overrides
    for key, val in overrides.items():
        keys = key.split('.')
        d = config
        for k in keys[:-1]:
            d = d[k]
        d[keys[-1]] = val

    return config

print("Config builder ready.")

---
## 4. Approach 1: TransUNet Regression

CNN encoder + Vision Transformer + U-Net decoder.  
**Loss:** L1 + MS-SSIM + Perceptual (combined).  
**Best for:** Baseline with good global context.

In [ ]:
from src.train import train_regression

config = make_config('transunet',
    **{
        'training.epochs': 100,
        'training.output_dir': '/content/drive/MyDrive/virtual_staining_outputs',
    }
)

print("Model:", config['model']['name'])
print("Image size:", config['data']['img_size'])
print("Batch size:", config['data']['batch_size'])
print("Epochs:", config['training']['epochs'])
print("Losses:", [l['name'] for l in config['loss']['losses']])

In [ ]:
# Train TransUNet
train_regression(config)

---
## 5. Approach 2: SwinUNet Regression

Window-based attention captures local details better than standard ViT patches.  
**Best for:** Detecting subtle, varying-intensity IF signals.

In [ ]:
from src.train import train_regression

config = make_config('swin_unet',
    **{
        'training.epochs': 100,
        'training.output_dir': '/content/drive/MyDrive/virtual_staining_outputs',
    }
)

train_regression(config)

---
## 6. Approach 3: Pix2PixHD (Conditional GAN)

Generator + multi-scale discriminator with feature matching.  
**Best for:** Sharper outputs, but needs careful training.  
**Note:** Needs more epochs (~200) and can be unstable.

In [ ]:
from src.train import train_gan

config = make_config('pix2pixhd',
    **{
        'training.epochs': 200,
        'training.output_dir': '/content/drive/MyDrive/virtual_staining_outputs',
    }
)

train_gan(config)

---
## 7. Approach 4: Conditional DDPM (Diffusion)

Conditioned on BF, iteratively denoises pure noise into IF images.  
**Best for:** Highest fidelity, least hallucination.  
**Tradeoff:** Slowest training + inference. Uses 256px by default to fit in memory.

**How it works:**
- Training: noise the ground truth IF at random timestep t, model learns to predict the noise conditioned on BF
- Inference: start from pure Gaussian noise, denoise step-by-step conditioned on BF
- DDIM sampling for faster inference (50 steps instead of 1000)

In [ ]:
from src.train import train_diffusion

config = make_config('diffusion',
    **{
        'data.img_size': 256,     # Diffusion is memory-heavy
        'data.batch_size': 2,     # Reduce for T4, can increase for A100
        'training.epochs': 300,
        'training.output_dir': '/content/drive/MyDrive/virtual_staining_outputs',
    }
)

train_diffusion(config)

---
## 8. Approach 5: AdaIN Style Transfer

Transfers IF "style" onto BF "content" using adaptive instance normalization.  
**Best for:** Domain adaptation (fixed BF → live BF) or fast style-based staining.  
**EFDM variant:** Set `use_efdm: true` for full distribution matching (higher fidelity).

In [ ]:
from src.train import train_style_transfer

config = make_config('adain',
    **{
        'training.epochs': 50,
        'training.output_dir': '/content/drive/MyDrive/virtual_staining_outputs',
    }
)

train_style_transfer(config)

---
## 9. Inference & Comparison

Load the best checkpoint from any model and run predictions on the test set.

In [ ]:
#@title Select model to evaluate { display-mode: "form" }
MODEL_NAME = "transunet"  #@param ["transunet", "swin_unet", "pix2pixhd", "diffusion", "adain"]
OUTPUT_DIR = "/content/drive/MyDrive/virtual_staining_outputs"  #@param {type:"string"}

import torch
import yaml
from src.models import build_model
from src.data import get_dataloaders
from src.utils import compute_metrics, MetricTracker

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load checkpoint
ckpt_path = f"{OUTPUT_DIR}/{MODEL_NAME}/best_model.pth"
ckpt = torch.load(ckpt_path, map_location=device)
config = ckpt['config']
print(f"Loaded {MODEL_NAME} from epoch {ckpt['epoch']}")
print(f"Val metrics at save: {ckpt.get('val_metrics', 'N/A')}")

# Build model and load weights
if MODEL_NAME == 'pix2pixhd':
    model_wrapper = build_model(config)
    model = model_wrapper.get_generator().to(device)
    model.load_state_dict(ckpt['generator_state'])
else:
    model = build_model(config).to(device)
    model.load_state_dict(ckpt['model_state'])

model.eval()
print("Model loaded.")

In [ ]:
# Run on test set
_, _, test_loader = get_dataloaders(config)
print(f"Test samples: {len(test_loader.dataset)}")

tracker = MetricTracker()
predictions = []

with torch.no_grad():
    for batch in test_loader:
        bf = batch['bf'].to(device)
        target = batch['target'].to(device)

        if MODEL_NAME == 'diffusion':
            pred = model.sample(bf, use_ddim=True, ddim_steps=50)
        elif MODEL_NAME == 'adain':
            pred = model.transfer(bf, target)  # Uses target as style ref
        else:
            pred = model(bf)

        pred = pred.clamp(0, 1)
        tracker.update(compute_metrics(pred, target))
        predictions.append((bf.cpu(), pred.cpu(), target.cpu(), batch['name']))

print(f"\nTest Results ({MODEL_NAME}):")
print(tracker)

In [ ]:
# Visualize predictions
import matplotlib.pyplot as plt
import numpy as np

n_show = min(5, len(predictions))
fig, axes = plt.subplots(n_show, 4, figsize=(16, 4 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]

def to_composite(t):
    """Convert 2-channel (DAPI, Phalloidin) to RGB composite."""
    if t.shape[0] == 2:
        rgb = np.zeros((*t.shape[1:], 3))
        rgb[:, :, 2] = t[0].numpy()  # DAPI = blue
        rgb[:, :, 0] = t[1].numpy()  # Phalloidin = red
        return rgb
    elif t.shape[0] == 1:
        return t[0].numpy()
    return t.permute(1, 2, 0).numpy()

for i in range(n_show):
    bf, pred, target, name = predictions[i]
    bf, pred, target = bf[0], pred[0], target[0]

    axes[i, 0].imshow(bf[0].numpy(), cmap='gray')
    axes[i, 0].set_title(f'BF Input')

    axes[i, 1].imshow(to_composite(target))
    axes[i, 1].set_title('Ground Truth IF')

    axes[i, 2].imshow(to_composite(pred))
    axes[i, 2].set_title(f'Predicted ({MODEL_NAME})')

    # Difference map
    diff = torch.abs(pred - target).mean(0).numpy()
    axes[i, 3].imshow(diff, cmap='hot', vmin=0, vmax=0.3)
    axes[i, 3].set_title('Error Map')

    for ax in axes[i]:
        ax.axis('off')

plt.suptitle(f'{MODEL_NAME} — Test Set Predictions', fontsize=14)
plt.tight_layout()
plt.show()

---
## 10. Compare All Models

Run this after training multiple models to get a side-by-side comparison.

In [ ]:
import pandas as pd

OUTPUT_DIR = "/content/drive/MyDrive/virtual_staining_outputs"
models_to_compare = ['transunet', 'swin_unet', 'pix2pixhd', 'diffusion', 'adain']

results = []
for name in models_to_compare:
    ckpt_path = f"{OUTPUT_DIR}/{name}/best_model.pth"
    try:
        ckpt = torch.load(ckpt_path, map_location='cpu')
        metrics = ckpt.get('val_metrics', {})
        results.append({
            'Model': name,
            'Epoch': ckpt.get('epoch', '?'),
            'PSNR': metrics.get('psnr', '-'),
            'SSIM': metrics.get('ssim', '-'),
            'MAE': metrics.get('mae', '-'),
            'PCC': metrics.get('pcc', '-'),
        })
    except FileNotFoundError:
        print(f"  {name}: not trained yet")

if results:
    df = pd.DataFrame(results)
    print(df.to_string(index=False))

---
## Tips & Troubleshooting

**Out of Memory (OOM):**
- Reduce `BATCH_SIZE` (try 2 or 1)
- Reduce `IMG_SIZE` (try 256)
- For diffusion: already defaults to 256px and batch_size=2
- Use Colab Pro for A100 GPU

**No image pairs found:**
- Check that your Drive folder has `project_*/BF/` and `project_*/DAPI/` subfolders
- Image filenames must share the same base name (minus `_chXX` suffix)

**Training is slow:**
- TransUNet/SwinUNet: ~2-3 min/epoch on T4 at 512px
- Pix2PixHD: ~3-5 min/epoch (G+D updates)
- Diffusion: ~5-10 min/epoch (most compute-heavy)
- AdaIN: ~1-2 min/epoch (fastest, decoder-only training)

**Outputs saved to:**
- `{OUTPUT_DIR}/{model_name}/best_model.pth` — best checkpoint
- `{OUTPUT_DIR}/{model_name}/samples/` — visual predictions per epoch
- `{OUTPUT_DIR}/{model_name}/config.yaml` — training config used